# 03 — Agentic RAG: Evidence Investigation with Tool Boundaries

**Track:** Advanced · **Stage:** Production Patterns

In Corrective RAG (CRAG), we built a deterministic state machine where the *path* was hardcoded (Retrieve -> Grade -> Fallback). 

In **Agentic RAG**, we provide the LLM with a set of **Tools** and allow it to dynamically determine its own path. The model reasons about the question, decides which tool to call, observes the output, and iterates until it has enough information to answer.

In this comprehensive deep dive, we will:
1. **Part 1: The Theory of Tool Boundaries.** Understand the difference between "Read" tools (safe) and "Execute" tools (dangerous), and how to enforce Human-in-the-Loop (HITL) approvals.
2. **Part 2: Production Implementation.** Build an autonomous ReAct agent using **LangGraph's `create_react_agent`**.

---
## Part 1: The Theory of Tool Boundaries

A production agent is not just a prompt; it requires strict boundaries. More agents do not fix weak retrieval or missing tool controls.

- **Read Tools** (e.g., Search Database) retrieve evidence. They are generally safe to run autonomously.
- **Execute Tools** (e.g., Rollback Deployment, Send Email) have real-world side effects. They must require typed inputs, authorization, and **Human-in-the-Loop (HITL) approval**.

In [ ]:
from typing import Dict, Any, Optional

class ToolRequest:
    def __init__(self, tool_name: str, args: Dict[str, Any], requires_approval: bool):
        self.tool_name = tool_name
        self.args = args
        self.requires_approval = requires_approval
        self.is_approved = False if requires_approval else True
        
    def execute(self) -> str:
        if not self.is_approved:
            return "ERROR: Execution denied. Human approval required."
        return f"SUCCESS: Executed {self.tool_name} with {self.args}"

# Scenario 1: A safe Read tool
read_tool = ToolRequest("search_knowledge_base", {"query": "checkout failure"}, requires_approval=False)
print("--- Executing Read Tool ---")
print(read_tool.execute())

# Scenario 2: A dangerous Execute tool proposed by the LLM
execute_tool = ToolRequest("rollback_deployment", {"deploy_id": "842"}, requires_approval=True)
print("\n--- Executing Dangerous Tool (Unapproved) ---")
print(execute_tool.execute())

# Scenario 3: Human approves the action
print("\n--- Executing Dangerous Tool (Approved) ---")
execute_tool.is_approved = True
print(execute_tool.execute())

---
## Part 2: Production Implementation with LangGraph

Now we build the agent. We will use the **ReAct (Reason + Act)** pattern. The LLM will loop:
1. **Thought:** What do I need to do?
2. **Action:** Call a tool.
3. **Observation:** Read the tool's output.
4. Repeat until the answer is found.

In [ ]:
# !pip install langgraph langchain langchain-core

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.prebuilt import create_react_agent
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration

### Step A: Define the Tools

We provide the agent with two tools. The LLM will decide which one to use.

In [ ]:
@tool
def internal_knowledge_search(query: str) -> str:
    """Search the internal secure company database."""
    print(f"  [Tool Execution] Searching internal DB for: {query}")
    if "deploy" in query:
        return "Deploy-842 caused a checkout failure. Mitigation: Rollback to 841."
    return "No internal docs found."

@tool
def web_search(query: str) -> str:
    """Search the public internet for general knowledge."""
    print(f"  [Tool Execution] Searching the web for: {query}")
    return "Web Result: Tacos are delicious."

tools = [internal_knowledge_search, web_search]

### Step B: Mocking the Agent LLM

For this offline tutorial, we mock an LLM that supports `tool_calls`. In production, you would use `ChatOpenAI(model="gpt-4o")` or similar.

In [ ]:
class MockToolCallingLLM(BaseChatModel):
    """A mock chat model that simulates returning tool calls."""
    
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        # Simple heuristic to mock the agent's behavior based on the message history length
        # If this is the first turn, call a tool.
        if len(messages) == 1:
            # Mocking a tool call to the internal DB
            tool_call = {
                "name": "internal_knowledge_search",
                "args": {"query": "deploy-842 status"},
                "id": "call_abc123"
            }
            msg = AIMessage(content="", tool_calls=[tool_call])
            return ChatResult(generations=[ChatGeneration(message=msg)])
        
        # If this is the second turn (after the tool responded), generate the final answer.
        msg = AIMessage(content="Based on the internal database, Deploy-842 caused a checkout failure. You should rollback to 841.")
        return ChatResult(generations=[ChatGeneration(message=msg)])
        
    @property
    def _llm_type(self) -> str:
        return "mock_tool_calling_llm"

    def bind_tools(self, tools, **kwargs):
        return self # Mock implementation ignores actual binding

# Instantiate the mock
llm = MockToolCallingLLM()

### Step C: Compile and Run the Agent

LangGraph's `create_react_agent` wires up the Reason -> Act -> Observe loop automatically.

In [ ]:
agent_executor = create_react_agent(llm, tools)

print("========== RUNNING AGENT ==========")
inputs = {"messages": [HumanMessage(content="Why is checkout failing after deploy-842?")]}

for chunk in agent_executor.stream(inputs):
    if "agent" in chunk:
        if chunk["agent"]["messages"][0].tool_calls:
            print(f"Agent Thought: I need to call a tool: {chunk['agent']['messages'][0].tool_calls[0]['name']}")
        else:
            print(f"\nAgent Final Answer: {chunk['agent']['messages'][0].content}")
            
    elif "tools" in chunk:
        print(f"Tool Output Observation: {chunk['tools']['messages'][0].content}")

## Reflection

1. **Traceability:** In Agentic RAG, tracing is critical. You must log *why* the agent chose a specific tool and what parameters it passed. Without traces, debugging a hallucinated tool call is impossible.
2. **State Management:** LangGraph `create_react_agent` automatically manages the `messages` array in the state, appending `HumanMessage`, `AIMessage` (with tool requests), and `ToolMessage` (with tool responses) so the LLM retains the full context of its investigation.